In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType
spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("day-23")
    
    # Use AWS profile credentials from ~/.aws/credentials
    # No access key or secret key stored in code
    .config(
        "spark.hadoop.fs.s3a.aws.credentials.provider",
        "com.amazonaws.auth.profile.ProfileCredentialsProvider"
    )
    
    # S3 connector packages
    .config(
        "spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.4.2,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.780"
    )
    
    .getOrCreate()
)

:: loading settings :: url = jar:file:/opt/homebrew/Cellar/apache-spark/4.1.1/libexec/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/rahulsinghrana/.ivy2.5.2/cache
The jars for the packages stored in: /Users/rahulsinghrana/.ivy2.5.2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-02c347c9-2c03-4604-a47e-98bbd5eb6743;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.4.2 in central
	found software.amazon.awssdk#bundle;2.29.52 in central
	found software.amazon.s3.analyticsaccelerator#analyticsaccelerator-s3;1.2.1 in central
	found org.wildfly.openssl#wildfly-openssl;2.1.4.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.780 in central
:: resolution report :: resolve 139ms :: artifacts dl 4ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.780 from central in [default]
	org.ap

In [2]:
customers_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/customers.csv')
orders_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/orders.csv')
products_df=spark.read.\
    option('header','true').\
        option('inferSchema','true').\
            csv('s3a://pyspark-30-days-rahul-2026/data/products.csv')

26/08/09 07:09:40 WARN CredentialProviderListFactory: Credentials option fs.s3a.aws.credentials.provider contains AWS v1 SDK entry com.amazonaws.auth.profile.ProfileCredentialsProvider; mapping to software.amazon.awssdk.auth.credentials.ProfileCredentialsProvider
SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


**Problem 1** | Easy

Orders with customer names

Return order_id, first_name, last_name, unit_price for every order, joined with the customer who placed it. Sort by unit_price descending, show the top 10.

In [3]:
orders_df1=orders_df.select('order_id','customer_id','unit_price')
customers_df1=customers_df.select('customer_id','first_name','last_name')
joined_df=orders_df1.join(customers_df1,on='customer_id',how='inner')
joined_df.orderBy(F.col('unit_price').desc()).show(10,truncate=False)

+-----------+--------+----------+----------+---------+
|customer_id|order_id|unit_price|first_name|last_name|
+-----------+--------+----------+----------+---------+
|C006       |O0051   |1299.99   |Patricia  |Davis    |
|C009       |O0009   |1299.99   |David     |Thomas   |
|C004       |O0024   |1299.99   |Linda     |Martinez |
|C014       |O0034   |1299.99   |Jessica   |Thompson |
|C021       |O0041   |1299.99   |Daniel    |Lee      |
|C001       |O0001   |1299.99   |James     |Anderson |
|C016       |O0061   |1299.99   |Sarah     |Martinez |
|C002       |O0072   |1299.99   |Maria     |Garcia   |
|C010       |O0080   |1299.99   |Jennifer  |Jackson  |
|C018       |O0088   |1299.99   |Karen     |Clark    |
+-----------+--------+----------+----------+---------+
only showing top 10 rows


**Problem 2** | Easy

Orders with product details

Return order_id, product_name, category, and the order's quantity for every order. Both orders and products have a unit_price column — handle the collision.

In [4]:
orders_df2=orders_df.select('order_id',
                            'product_id',
                            'unit_price',
                            'quantity').alias('o')
products_df2=products_df.select('product_id',
                                'product_name',
                                'category').alias('p')
joined_df2=orders_df2.join(products_df2,on='product_id',how='inner')
joined_df2.select('o.order_id',
                  'p.product_name',
                  'p.category',
                  'o.quantity',
                  'o.unit_price',
                  ).\
    orderBy(F.col('o.unit_price').desc()).show(10,truncate=False)

+--------+-------------+-----------+--------+----------+
|order_id|product_name |category   |quantity|unit_price|
+--------+-------------+-----------+--------+----------+
|O0051   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0009   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0024   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0034   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0041   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0001   |Laptop Pro 15|Electronics|2       |1299.99   |
|O0061   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0072   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0080   |Laptop Pro 15|Electronics|1       |1299.99   |
|O0088   |Laptop Pro 15|Electronics|1       |1299.99   |
+--------+-------------+-----------+--------+----------+
only showing top 10 rows


**Problem 3** | Medium

Customers who have never ordered

Find every customer in customers who has zero orders in orders. Return customer_id, first_name, last_name. This is a classic "find what's missing" join problem.

In [5]:
customers_df3 = customers_df.select(
    'customer_id',
    'first_name',
    'last_name'
).alias('c')

orders_df3 = orders_df.select(
    'customer_id'
).alias('o')

joined_df3 = customers_df3.join(
    orders_df3,
    on='customer_id',
    how='left_anti'
)

joined_df3.show(10, truncate=False)

+-----------+----------+---------+
|customer_id|first_name|last_name|
+-----------+----------+---------+
+-----------+----------+---------+



**Problem 4** | Medium

Products that have never been ordered

Find every product in products that does not appear in any row of orders. Return product_id, product_name, category. Same pattern as Problem 3, different tables.

In [6]:
products_df4 = products_df.select(
    'product_id',
    'product_name',
    'category'
).alias('p')

orders_df4 = orders_df.select(
    'product_id'
).alias('o')

joined_df4 = products_df4.join(
    orders_df4,
    on='product_id',
    how='left_anti'
)

joined_df4.select(
    'p.product_id',
    'p.product_name',
    'p.category'
).show(10, truncate=False)

+----------+------------------+---------------+
|product_id|product_name      |category       |
+----------+------------------+---------------+
|P015      |Printer All-in-One|Electronics    |
|P016      |Paper Shredder    |Office Supplies|
|P018      |Filing Cabinet    |Furniture      |
+----------+------------------+---------------+



**Problem 5** | Hard

Three-way join with aggregation

Join orders, customers, and products together. For each segment + category combination, calculate total revenue (order.unit_price * quantity). Sort by revenue descending, show the top 10 combinations.

In [9]:
orders_df4 = orders_df.select(
    "customer_id",
    "product_id",
    "unit_price",
    "quantity"
)

customers_df4 = customers_df.select(
    "customer_id",
    "segment"
)

products_df4 = products_df.select(
    "product_id",
    "category"
)

joined_df = (
    orders_df4
    .join(customers_df4, on="customer_id", how="inner")
    .join(products_df4, on="product_id", how="inner")
)

result_df = (
    joined_df
    .withColumn(
        "revenue",
        F.col("unit_price") * F.col("quantity")
    )
    .groupBy("segment", "category")
    .agg(
        F.round(F.sum("revenue"),2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
    .limit(10)
)

result_df.show(truncate=False)

+----------+---------------+-------------+
|segment   |category       |total_revenue|
+----------+---------------+-------------+
|Enterprise|Electronics    |14439.25     |
|SMB       |Electronics    |12159.49     |
|Startup   |Electronics    |8489.63      |
|Enterprise|Furniture      |4109.72      |
|Startup   |Furniture      |2669.88      |
|SMB       |Furniture      |2229.79      |
|Startup   |Office Supplies|154.93       |
|SMB       |Office Supplies|129.99       |
+----------+---------------+-------------+



**Problem 6** | Hard

Self-referencing comparison — customers in the same city

For every pair of different customers who live in the same city, return both customers' customer_id and city — without duplicate pairs (i.e. if A-B is returned, B-A should not be). This requires joining a table to itself.

In [15]:
c1 = customers_df.select(
    F.col("customer_id").alias("customer1"),
    F.col("city").alias("city1")
).distinct()

c2 = customers_df.select(
    F.col("customer_id").alias("customer2"),
    F.col("city").alias("city2")
).distinct()

result = (
    c1.join(
        c2,
        (F.col("city1") == F.col("city2")) &
        (F.col("customer1") < F.col("customer2")),
        "inner"
    )
    .select(
        F.col("customer1"),
        F.col("customer2"),
        F.col("city1").alias("city")
    )
)
result.show(truncate=False)

+---------+---------+----+
|customer1|customer2|city|
+---------+---------+----+
+---------+---------+----+

